## LangGraph检查点Checkpoint字段含义剖析

In [1]:
# 必须实现per-thread subgraphs in tool以实现查看子图的状态！
import uuid

from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from rich.pretty import pprint

from src.config.llm import LLMs
from src.utils.graph_running import multi_turn_conversation


@tool
def fruit_info(fruit_name: str) -> str:
    """Look up fruit info."""
    return f"Info about {fruit_name}"


@tool
def veggie_info(veggie_name: str) -> str:
    """Look up veggie info."""
    return f"Info about {veggie_name}"


from langgraph.graph import MessagesState, StateGraph


def create_sub_agent(model, *, name, **kwargs):
    """Wrap an agent with a unique node name for namespace isolation."""
    agent = create_agent(model=model, name=name, **kwargs)
    #  when parent graph and subgraph share state keys, you can pass a compiled subgraph directly to add_node.
    #  the subgraph reads from and writes to the parent’s state channels automatically
    return (
        StateGraph(MessagesState)
        .add_node(f"ns_for_{name}", agent)  # unique name → stable namespace  #
        .add_edge("__start__", f"ns_for_{name}")
        .compile(name=f"Graph_with_one_{name}_node")  # 若传入的kwargs中已经有checkpointer=True，则此处不必checkpointer=True
    )


fruit_agent = create_sub_agent(
    LLMs["minimax-m2.7"],
    name="fruit_agent",
    tools=[fruit_info],
    system_prompt="You are a fruit expert. Use the fruit_info tool. Respond in one short sentence every time.",
    checkpointer=True,  # 开启per-thread的子图状态，确保副智能体支持多轮对话的记忆
)
veggie_agent = create_sub_agent(
    LLMs["minimax-m2.7"],
    name="veggie_agent",
    tools=[veggie_info],
    system_prompt="You are a veggie expert. Use the veggie_info tool. Respond in one short sentence every time.",
    checkpointer=True,  # 开启per-thread的子图状态，确保副智能体支持多轮对话的记忆
)


# Wrap subagents as tools for the outer agent
@tool
def ask_fruit_expert(question: str) -> str:
    """Ask the fruit expert. Use for ALL fruit questions."""
    response = fruit_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
    )
    return response["messages"][-1].content


@tool
def ask_veggie_expert(question: str) -> str:
    """Ask the veggie expert. Use for ALL veggie questions."""
    response = veggie_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
    )
    return response["messages"][-1].content


# Outer agent with checkpointer
agent = create_agent(
    model=LLMs["minimax-m2.7"],
    tools=[ask_fruit_expert, ask_veggie_expert],
    system_prompt=(
        "You have two experts: ask_fruit_expert and ask_veggie_expert. "
        "ALWAYS delegate questions to the appropriate expert."
    ),
    middleware=[
        ToolCallLimitMiddleware(tool_name="ask_fruit_expert", run_limit=1),  # 不允许同时多次调用同一个per-thread类型的子图
        ToolCallLimitMiddleware(tool_name="ask_veggie_expert", run_limit=1),  # 不允许同时多次调用同一个per-thread类型的子图
    ],
    checkpointer=MemorySaver(),
)

config: RunnableConfig = {
    "configurable":
        {"thread_id": str(uuid.uuid4())}
}

# First call - LLM calls both fruit and veggie experts
response_1 = agent.invoke(
    {"messages": [{"role": "user", "content": "跟我简单讲一讲樱桃和西兰花"}]},
    # {"messages": [{"role": "user", "content": "跟我简单讲一讲樱桃"}]},
    config=config,
)
# Fruit subagent message count: 4
# Veggie subagent message count: 4

# agent_state = agent.get_state(config, subgraphs=True)
# pprint(agent_state)

agent_states = list(agent.get_state_history(config))
pprint(agent_states)


[
│   StateSnapshot(
│   │   values={
│   │   │   'messages': [
│   │   │   │   HumanMessage(
│   │   │   │   │   content='跟我简单讲一讲樱桃',
│   │   │   │   │   additional_kwargs={},
│   │   │   │   │   response_metadata={},
│   │   │   │   │   id='a7be5c3b-466d-421c-8f87-ed4491aea413'
│   │   │   │   ),
│   │   │   │   AIMessage(
│   │   │   │   │   content='',
│   │   │   │   │   additional_kwargs={'refusal': None},
│   │   │   │   │   response_metadata={
│   │   │   │   │   │   'token_usage': {
│   │   │   │   │   │   │   'completion_tokens': 57,
│   │   │   │   │   │   │   'prompt_tokens': 265,
│   │   │   │   │   │   │   'total_tokens': 322,
│   │   │   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   │   │   'accepted_prediction_tokens': None,
│   │   │   │   │   │   │   │   'audio_tokens': None,
│   │   │   │   │   │   │   │   'reasoning_tokens': 0,
│   │   │   │   │   │   │   │   'rejected_prediction_tokens': None
│   │   │   │   │   │   │   },
│   │   │   │   │   │   │   'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
│   │   │   │   │   │   │   'total_characters': 0
│   │   │   │   │   │   },
│   │   │   │   │   │   'model_provider': 'openai',
│   │   │   │   │   │   'model_name': 'MiniMax-M2.7',
│   │   │   │   │   │   'system_fingerprint': None,
│   │   │   │   │   │   'id': '06723a716369306f855d782bd71862a1',
│   │   │   │   │   │   'finish_reason': 'tool_calls',
│   │   │   │   │   │   'logprobs': None
│   │   │   │   │   },
│   │   │   │   │   id='lc_run--019e98d5-884c-7ad0-b334-f370b5337544-0',
│   │   │   │   │   tool_calls=[
│   │   │   │   │   │   {
│   │   │   │   │   │   │   'name': 'ask_fruit_expert',
│   │   │   │   │   │   │   'args': {
│   │   │   │   │   │   │   │   'question': '请用简单的语言介绍一下樱桃，包括它的基本特点、营养价值和常见品种。'
│   │   │   │   │   │   │   },
│   │   │   │   │   │   │   'id': 'chatcmpl-tool-ab00837d0ae790fc',
│   │   │   │   │   │   │   'type': 'tool_call'
│   │   │   │   │   │   }
│   │   │   │   │   ],
│   │   │   │   │   invalid_tool_calls=[],
│   │   │   │   │   usage_metadata={
│   │   │   │   │   │   'input_tokens': 265,
│   │   │   │   │   │   'output_tokens': 57,
│   │   │   │   │   │   'total_tokens': 322,
│   │   │   │   │   │   'input_token_details': {'cache_read': 0},
│   │   │   │   │   │   'output_token_details': {'reasoning': 0}
│   │   │   │   │   }
│   │   │   │   ),
│   │   │   │   ToolMessage(
│   │   │   │   │   content='樱桃是一种小巧圆润的红色水果，味道甜美多汁。它富含维生素C、维生素A、钾、铁等营养，具有抗氧化和补血功效。常见品种有中国樱桃和甜樱桃（车厘子）。',
│   │   │   │   │   name='ask_fruit_expert',
│   │   │   │   │   id='be0e9f68-857b-4d17-b2b5-01642c34e92e',
│   │   │   │   │   tool_call_id='chatcmpl-tool-ab00837d0ae790fc'
│   │   │   │   ),
│   │   │   │   AIMessage(
│   │   │   │   │   content='\n樱桃是一种小巧圆润的红色水果，味道甜美多汁。它富含维生素C、维生素A、钾、铁等营养成分，具有抗氧化和补血的功效。常见品种有中国樱桃和甜樱桃（车厘子）。',
│   │   │   │   │   additional_kwargs={'refusal': None},
│   │   │   │   │   response_metadata={
│   │   │   │   │   │   'token_usage': {
│   │   │   │   │   │   │   'completion_tokens': 61,
│   │   │   │   │   │   │   'prompt_tokens': 363,
│   │   │   │   │   │   │   'total_tokens': 424,
│   │   │   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   │   │   'accepted_prediction_tokens': None,
│   │   │   │   │   │   │   │   'audio_tokens': None,
│   │   │   │   │   │   │   │   'reasoning_tokens': 0,
│   │   │   │   │   │   │   │   'rejected_prediction_tokens': None
│   │   │   │   │   │   │   },
│   │   │   │   │   │   │   'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 174},
│   │   │   │   │   │   │   'total_characters': 0
│   │   │   │   │   │   },
│   │   │   │   │   │   'model_provider': 'openai',
│   │   │   │   │   │   'model_name': 'MiniMax-M2.7',
│   │   │   │   │   │   'system_fingerprint': None,
│   │   │   │   │   │   'id': '06723a781c6a39d6a1ef1fdad40e373f',
│   │   │   │   │   │   'finish_reason': 'stop',
│   │   │   │   │   │   'logprobs': None
│   │   │   │   │   },
│   │   │   

### **正确的执行时序**

LangGraph 的 checkpoint 保存发生在**任务执行之后**，但采用了一种特殊的结构设计：

```
迭代 N 的完整流程：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 从 Checkpoint N-1 读取
   - values: 上一轮的状态
   - next: 本轮要执行的节点列表

2. 执行所有 next 节点
   - 真正调用 LLM、工具等
   - 产生输出结果

3. 【关键步骤】构建 Checkpoint N
   ├─ values = 上一轮的 state（尚未合并本轮结果），可以认为是本轮任务的输入
   ├─ next = 本轮要执行的一批节点（任务）
   └─ tasks = 本轮已执行完毕的节点（任务）的其返回结果

4. 保存 Checkpoint N 到数据库/内存
```


### **为什么这样设计？**

这种设计的关键在于：**`values` 和 `tasks[].result` 之间存在"时间差"**

- **`values`**: 表示"在执行本轮任务**之前**的状态"
- **`tasks[].result`**: 表示"本轮任务的执行结果"

这样做的好处：

1. **可追溯性**：你可以清楚地看到"从这个状态出发，执行了哪些任务，产生了什么结果"
2. **中断恢复**：如果在中途暂停，你可以知道"已经执行了什么，结果是什么，下一步该做什么"
3. **调试友好**：可以 inspect 每个 checkpoint 的因果关系

### **回到你的例子**

```python
# 快照2 (step=0) 的真实含义：
StateSnapshot(
    values={
        # 这是执行 model 节点之前的状态
        'messages': [HumanMessage(...)]
    },
    next=('model',),  # 本轮要执行的节点是 model
    tasks=(
        PregelTask(
            name='model',
            result={  # model 节点已经执行完毕，这是它的输出
                'messages': [AIMessage(..., tool_calls=[...])]
            }
        ),
    )
)

# 当你从这个 checkpoint 恢复执行时（此时叫“重播”而不是分支，因为重播只能让你再次看到已经播放过的画面）：
# 1. LangGraph 会读取 tasks[0].result
# 2. 将其合并到 values
# 3. 得到新的 state: {messages: [HumanMessage, AIMessage]}
# 4. 然后根据 next 确定下一批要执行的节点
```


### **总结回答你的问题**

> "保存检查点时还未执行未来节点，保存时怎么知道 tasks 中每个 task 的 result 属性要填入什么值呢？"

**答案**：Checkpoint **不是在任务执行前保存**，而是在**任务执行后保存**。`tasks[].result` 填充的是**刚刚执行完毕的任务的结果**，而不是"未来节点的结果"。

**LangGraph 在每轮迭代的任务执行完毕后保存 checkpoint，其中：**
- **`values`** = 开始执行本轮任务之前的状态（历史状态）
- **`next`** = 本轮要执行的节点
- **`tasks[].result`** = 本轮要执行的节点的执行结果（刚刚产生的）

In [2]:
# Second call - both agents accumulate independently
response_2 = agent.invoke(
    # {"messages": [{"role": "user", "content": "现在跟我简单讲一讲橙子和胡萝卜"}]},
    {"messages": [{"role": "user", "content": "水果专家是否正常工作？"}]},
    config=config,
)
# Fruit subagent message count: 8 (remembers cherries!)
# Veggie subagent message count: 8 (remembers broccoli!)

# agent_state = agent.get_state(config, subgraphs=True)
# pprint(agent_state)

agent_states = list(agent.get_state_history(config))
pprint(agent_states)

[
│   StateSnapshot(
│   │   values={
│   │   │   'messages': [
│   │   │   │   HumanMessage(
│   │   │   │   │   content='跟我简单讲一讲樱桃',
│   │   │   │   │   additional_kwargs={},
│   │   │   │   │   response_metadata={},
│   │   │   │   │   id='a7be5c3b-466d-421c-8f87-ed4491aea413'
│   │   │   │   ),
│   │   │   │   AIMessage(
│   │   │   │   │   content='',
│   │   │   │   │   additional_kwargs={'refusal': None},
│   │   │   │   │   response_metadata={
│   │   │   │   │   │   'token_usage': {
│   │   │   │   │   │   │   'completion_tokens': 57,
│   │   │   │   │   │   │   'prompt_tokens': 265,
│   │   │   │   │   │   │   'total_tokens': 322,
│   │   │   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   │   │   'accepted_prediction_tokens': None,
│   │   │   │   │   │   │   │   'audio_tokens': None,
│   │   │   │   │   │   │   │   'reasoning_tokens': 0,
│   │   │   │   │   │   │   │   'rejected_prediction_tokens': None
│   │   │   │   │   │   │   },
│   │   │   │   │   │   │   'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
│   │   │   │   │   │   │   'total_characters': 0
│   │   │   │   │   │   },
│   │   │   │   │   │   'model_provider': 'openai',
│   │   │   │   │   │   'model_name': 'MiniMax-M2.7',
│   │   │   │   │   │   'system_fingerprint': None,
│   │   │   │   │   │   'id': '06723a716369306f855d782bd71862a1',
│   │   │   │   │   │   'finish_reason': 'tool_calls',
│   │   │   │   │   │   'logprobs': None
│   │   │   │   │   },
│   │   │   │   │   id='lc_run--019e98d5-884c-7ad0-b334-f370b5337544-0',
│   │   │   │   │   tool_calls=[
│   │   │   │   │   │   {
│   │   │   │   │   │   │   'name': 'ask_fruit_expert',
│   │   │   │   │   │   │   'args': {
│   │   │   │   │   │   │   │   'question': '请用简单的语言介绍一下樱桃，包括它的基本特点、营养价值和常见品种。'
│   │   │   │   │   │   │   },
│   │   │   │   │   │   │   'id': 'chatcmpl-tool-ab00837d0ae790fc',
│   │   │   │   │   │   │   'type': 'tool_call'
│   │   │   │   │   │   }
│   │   │   │   │   ],
│   │   │   │   │   invalid_tool_calls=[],
│   │   │   │   │   usage_metadata={
│   │   │   │   │   │   'input_tokens': 265,
│   │   │   │   │   │   'output_tokens': 57,
│   │   │   │   │   │   'total_tokens': 322,
│   │   │   │   │   │   'input_token_details': {'cache_read': 0},
│   │   │   │   │   │   'output_token_details': {'reasoning': 0}
│   │   │   │   │   }
│   │   │   │   ),
│   │   │   │   ToolMessage(
│   │   │   │   │   content='樱桃是一种小巧圆润的红色水果，味道甜美多汁。它富含维生素C、维生素A、钾、铁等营养，具有抗氧化和补血功效。常见品种有中国樱桃和甜樱桃（车厘子）。',
│   │   │   │   │   name='ask_fruit_expert',
│   │   │   │   │   id='be0e9f68-857b-4d17-b2b5-01642c34e92e',
│   │   │   │   │   tool_call_id='chatcmpl-tool-ab00837d0ae790fc'
│   │   │   │   ),
│   │   │   │   AIMessage(
│   │   │   │   │   content='\n樱桃是一种小巧圆润的红色水果，味道甜美多汁。它富含维生素C、维生素A、钾、铁等营养成分，具有抗氧化和补血的功效。常见品种有中国樱桃和甜樱桃（车厘子）。',
│   │   │   │   │   additional_kwargs={'refusal': None},
│   │   │   │   │   response_metadata={
│   │   │   │   │   │   'token_usage': {
│   │   │   │   │   │   │   'completion_tokens': 61,
│   │   │   │   │   │   │   'prompt_tokens': 363,
│   │   │   │   │   │   │   'total_tokens': 424,
│   │   │   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   │   │   'accepted_prediction_tokens': None,
│   │   │   │   │   │   │   │   'audio_tokens': None,
│   │   │   │   │   │   │   │   'reasoning_tokens': 0,
│   │   │   │   │   │   │   │   'rejected_prediction_tokens': None
│   │   │   │   │   │   │   },
│   │   │   │   │   │   │   'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 174},
│   │   │   │   │   │   │   'total_characters': 0
│   │   │   │   │   │   },
│   │   │   │   │   │   'model_provider': 'openai',
│   │   │   │   │   │   'model_name': 'MiniMax-M2.7',
│   │   │   │   │   │   'system_fingerprint': None,
│   │   │   │   │   │   'id': '06723a781c6a39d6a1ef1fdad40e373f',
│   │   │   │   │   │   'finish_reason': 'stop',
│   │   │   │   │   │   'logprobs': None
│   │   │   │   │   },
│   │   │   

## 第二次运行agent，发现本次运行依然会从__start__节点开始执行，并且该节点的返回结果就是往messages列表中加入最新的人类消息“水果专家是否能正常工作？”

## 但是我们如何获取水果专家智能体的内部消息状态呢？目前statssnapshot中的tasks仅仅能展示出专家智能体作为tool被执行后对于agent状态的改变。

In [ ]:
multi_turn_conversation(agent, config)